In [1]:
"""
Cell 1 — Load raw eGeMAPS CSVs, compute 7-statistic participant-level
aggregation (mean, std, median, min, max, p25, p75) per feature.
Official E-DAIC split: train/dev/test are kept exactly as given.
"""

import numpy as np
import pandas as pd
from pathlib import Path

DATASET_ROOT = Path("/kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset")
RAW_DIR = DATASET_ROOT / "raw_egemaps"
LABELS_DIR = DATASET_ROOT / "labels"

STATS = ["mean", "std", "median", "min", "max", "p25", "p75"]

def egemaps_path(pid):
    matches = list((RAW_DIR / f"{pid}_P").glob("*egemaps.csv"))
    return matches[0] if matches else None

def compute_participant_stats(arr):
    """arr: (n_frames, 23) raw feature matrix -> (161,) stat vector, NaN/inf-safe."""
    arr = np.where(np.isfinite(arr), arr, np.nan)  # inf -> nan, so nan-aware stats handle both
    stats = [
        np.nanmean(arr, axis=0),
        np.nanstd(arr, axis=0),
        np.nanmedian(arr, axis=0),
        np.nanmin(arr, axis=0),
        np.nanmax(arr, axis=0),
        np.nanpercentile(arr, 25, axis=0),
        np.nanpercentile(arr, 75, axis=0),
    ]
    vec = np.concatenate(stats)
    vec = np.where(np.isfinite(vec), vec, 0.0)  # any all-NaN column -> 0.0 fallback
    return vec

def load_split(split_name):
    labels = pd.read_csv(LABELS_DIR / f"{split_name}_split.csv")[["Participant_ID", "PHQ_Binary"]]
    rows, kept_pids, kept_labels = [], [], []
    feature_cols = None

    for _, row in labels.iterrows():
        pid, y = row["Participant_ID"], row["PHQ_Binary"]
        p = egemaps_path(pid)
        if p is None:
            print(f"[warn] missing raw file for participant {pid} ({split_name}) — skipped")
            continue
        d = pd.read_csv(p, sep=";")
        if feature_cols is None:
            feature_cols = [c for c in d.columns if c not in ("name", "frameTime")]
            assert len(feature_cols) == 23, f"Expected 23 feature columns, got {len(feature_cols)}"
        arr = d[feature_cols].values.astype(np.float64)
        rows.append(compute_participant_stats(arr))
        kept_pids.append(pid)
        kept_labels.append(y)

    feat_names = [f"{fc}_{s}" for s in STATS for fc in feature_cols]
    X = pd.DataFrame(rows, columns=feat_names)
    X.insert(0, "Participant_ID", kept_pids)
    X["PHQ_Binary"] = kept_labels
    return X

print("Loading train/dev/test — this reads all 275 raw eGeMAPS CSVs, may take a few minutes...")
train_data = load_split("train")
dev_data = load_split("dev")
test_data = load_split("test")

n_features = len([c for c in train_data.columns if c not in ("Participant_ID", "PHQ_Binary")])
print(f"\nFeature vector length per participant: {n_features} (expected 23 x 7 = 161)")
print(f"Train: {len(train_data)} | Dev: {len(dev_data)} | Test: {len(test_data)}")
print("\nTrain class distribution:", train_data.PHQ_Binary.value_counts().to_dict())
print("Dev class distribution:", dev_data.PHQ_Binary.value_counts().to_dict())
print("Test class distribution:", test_data.PHQ_Binary.value_counts().to_dict())

Loading train/dev/test — this reads all 275 raw eGeMAPS CSVs, may take a few minutes...

Feature vector length per participant: 161 (expected 23 x 7 = 161)
Train: 163 | Dev: 56 | Test: 56

Train class distribution: {0: 126, 1: 37}
Dev class distribution: {0: 44, 1: 12}
Test class distribution: {0: 39, 1: 17}


In [2]:
"""
Cell 2 — Impute/scale (fit on TRAIN only), tune SVM(RBF) via
StratifiedKFold on TRAIN only. No dev/test leakage into fitting.
"""

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV

RANDOM_STATE = 42
feature_cols = [c for c in train_data.columns if c not in ("Participant_ID", "PHQ_Binary")]

X_train_raw = train_data[feature_cols].values
X_dev_raw = dev_data[feature_cols].values
X_test_raw = test_data[feature_cols].values
y_train = train_data["PHQ_Binary"].values
y_dev = dev_data["PHQ_Binary"].values
y_test = test_data["PHQ_Binary"].values

# ------------------------------------------------------------------
# Imputer + scaler fitted on TRAIN ONLY — applied (transform only) to dev/test
# ------------------------------------------------------------------
imputer = SimpleImputer(strategy="median").fit(X_train_raw)
X_train_imp = imputer.transform(X_train_raw)
X_dev_imp = imputer.transform(X_dev_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler().fit(X_train_imp)
X_train = scaler.transform(X_train_imp)
X_dev = scaler.transform(X_dev_imp)
X_test = scaler.transform(X_test_imp)

# ------------------------------------------------------------------
# SVM (RBF) hyperparameter search — TRAIN-only StratifiedKFold, no dev/test involved
# ------------------------------------------------------------------
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.001],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

svm_base = SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE)
grid_search = GridSearchCV(svm_base, param_grid, scoring="f1", cv=cv, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_svm = grid_search.best_estimator_
best_params = grid_search.best_params_
print("Best hyperparameters (train-CV, F1-scored):", best_params)
print(f"Best train-CV F1: {grid_search.best_score_:.4f}")

Best hyperparameters (train-CV, F1-scored): {'C': 1, 'gamma': 0.001}
Best train-CV F1: 0.3182


In [3]:
"""
Cell 3 — Threshold selection on DEV ONLY (0.10-0.90 sweep, pick by DEV F1),
freeze, apply identically to TEST. Full metric suite on both splits.
Test is touched only at the very end, at the frozen threshold.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix)

# ------------------------------------------------------------------
# Probability scores (continuous, for ROC-AUC/PR-AUC — threshold-independent)
# ------------------------------------------------------------------
dev_proba = best_svm.predict_proba(X_dev)[:, 1]
test_proba = best_svm.predict_proba(X_test)[:, 1]

dev_roc_auc = roc_auc_score(y_dev, dev_proba)
dev_pr_auc = average_precision_score(y_dev, dev_proba)
test_roc_auc = roc_auc_score(y_test, test_proba)
test_pr_auc = average_precision_score(y_test, test_proba)

# ------------------------------------------------------------------
# Threshold sweep on DEV ONLY — selected purely by DEV F1, TEST untouched here
# ------------------------------------------------------------------
thresholds = np.arange(0.10, 0.91, 0.01)
best_threshold, best_dev_f1 = 0.5, -1
threshold_log = []

for t in thresholds:
    preds = (dev_proba >= t).astype(int)
    f1 = f1_score(y_dev, preds, zero_division=0)
    threshold_log.append((round(t, 2), f1))
    if f1 > best_dev_f1:
        best_dev_f1, best_threshold = f1, t

print(f"Best DEV threshold: {best_threshold:.2f} (DEV F1 at this threshold: {best_dev_f1:.4f})")

# ------------------------------------------------------------------
# Frozen threshold applied identically to DEV (for reporting) and TEST
# ------------------------------------------------------------------
def compute_metrics(y_true, proba, threshold):
    preds = (proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "ConfusionMatrix": confusion_matrix(y_true, preds).tolist(),
    }

dev_metrics = compute_metrics(y_dev, dev_proba, best_threshold)
dev_metrics["ROC_AUC"] = dev_roc_auc
dev_metrics["PR_AUC"] = dev_pr_auc

test_metrics = compute_metrics(y_test, test_proba, best_threshold)  # TEST used only here, at frozen threshold
test_metrics["ROC_AUC"] = test_roc_auc
test_metrics["PR_AUC"] = test_pr_auc

# ------------------------------------------------------------------
# Compact experiment summary — BTP results table format
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY — SVM (RBF) on 7-stat aggregated eGeMAPS")
print("=" * 70)
print(f"Feature vector length     : {n_features}  (23 eGeMAPS features x 7 stats)")
print(f"Train / Dev / Test counts : {len(train_data)} / {len(dev_data)} / {len(test_data)}")
print(f"Train class distribution  : {train_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Dev class distribution    : {dev_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Test class distribution   : {test_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Best hyperparameters      : {best_params}")
print(f"Best DEV threshold        : {best_threshold:.2f}")
print("-" * 70)

summary_df = pd.DataFrame([
    {"Split": "DEV", **{k: v for k, v in dev_metrics.items() if k != "ConfusionMatrix"}},
    {"Split": "TEST", **{k: v for k, v in test_metrics.items() if k != "ConfusionMatrix"}},
])
print(summary_df.round(4).to_string(index=False))

print("-" * 70)
print("DEV  confusion matrix [[TN FP] [FN TP]]:", dev_metrics["ConfusionMatrix"])
print("TEST confusion matrix [[TN FP] [FN TP]]:", test_metrics["ConfusionMatrix"])
print("=" * 70)

summary_df.to_csv("/kaggle/working/svm_7stat_experiment_summary.csv", index=False)

Best DEV threshold: 0.10 (DEV F1 at this threshold: 0.3529)

EXPERIMENT SUMMARY — SVM (RBF) on 7-stat aggregated eGeMAPS
Feature vector length     : 161  (23 eGeMAPS features x 7 stats)
Train / Dev / Test counts : 163 / 56 / 56
Train class distribution  : {0: 126, 1: 37}
Dev class distribution    : {0: 44, 1: 12}
Test class distribution   : {0: 39, 1: 17}
Best hyperparameters      : {'C': 1, 'gamma': 0.001}
Best DEV threshold        : 0.10
----------------------------------------------------------------------
Split  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
  DEV    0.2143     0.2143     1.0 0.3529   0.5739  0.3163
 TEST    0.3036     0.3036     1.0 0.4658   0.6621  0.4523
----------------------------------------------------------------------
DEV  confusion matrix [[TN FP] [FN TP]]: [[0, 44], [0, 12]]
TEST confusion matrix [[TN FP] [FN TP]]: [[0, 39], [0, 17]]


# LOGISTIC REGRESSION

In [4]:
"""
Cell 2 — Preprocess (impute + scale, TRAIN-fit only), compact C search
via StratifiedKFold on TRAIN only, for two variants: no-PCA (primary)
and with-PCA (comparison). PCA, when used, is also fit on TRAIN only.
"""

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV

RANDOM_STATE = 42

X_train_raw = train_data[feature_cols].values
X_dev_raw = dev_data[feature_cols].values
X_test_raw = test_data[feature_cols].values
y_train = train_data["PHQ_Binary"].values
y_dev = dev_data["PHQ_Binary"].values
y_test = test_data["PHQ_Binary"].values

# Impute — fit on TRAIN only
imputer = SimpleImputer(strategy="median").fit(X_train_raw)
X_train_imp = imputer.transform(X_train_raw)
X_dev_imp = imputer.transform(X_dev_raw)
X_test_imp = imputer.transform(X_test_raw)

# Scale — fit on TRAIN only
scaler = StandardScaler().fit(X_train_imp)
X_train_scaled = scaler.transform(X_train_imp)
X_dev_scaled = scaler.transform(X_dev_imp)
X_test_scaled = scaler.transform(X_test_imp)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
C_grid = {"C": [0.001, 0.01, 0.1, 1, 10, 100]}

def tune_logreg(X_tr, y_tr):
    base = LogisticRegression(class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
    search = GridSearchCV(base, C_grid, scoring="f1", cv=cv, n_jobs=-1)
    search.fit(X_tr, y_tr)
    return search.best_estimator_, search.best_params_, search.best_score_

# ------------------------------------------------------------------
# Variant A: no PCA (primary model)
# ------------------------------------------------------------------
print("Tuning Logistic Regression WITHOUT PCA...")
model_no_pca, params_no_pca, cvf1_no_pca = tune_logreg(X_train_scaled, y_train)
print(f"  Best C: {params_no_pca['C']}  | train-CV F1: {cvf1_no_pca:.4f}")

# ------------------------------------------------------------------
# Variant B: with PCA (comparison only) — PCA fit on TRAIN only
# ------------------------------------------------------------------
print("\nTuning Logistic Regression WITH PCA (comparison)...")
pca = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(X_train_scaled)
X_train_pca = pca.transform(X_train_scaled)
X_dev_pca = pca.transform(X_dev_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"  PCA components retained (95% variance): {pca.n_components_} / {n_features}")

model_pca, params_pca, cvf1_pca = tune_logreg(X_train_pca, y_train)
print(f"  Best C: {params_pca['C']}  | train-CV F1: {cvf1_pca:.4f}")

print(f"\nTrain-CV F1 comparison — no-PCA: {cvf1_no_pca:.4f} vs with-PCA: {cvf1_pca:.4f}")
print("Proceeding with the NO-PCA model as primary (per request); "
      "PCA variant kept for reporting/comparison only.")

Tuning Logistic Regression WITHOUT PCA...
  Best C: 0.01  | train-CV F1: 0.3065

Tuning Logistic Regression WITH PCA (comparison)...
  PCA components retained (95% variance): 35 / 161
  Best C: 100  | train-CV F1: 0.3527

Train-CV F1 comparison — no-PCA: 0.3065 vs with-PCA: 0.3527
Proceeding with the NO-PCA model as primary (per request); PCA variant kept for reporting/comparison only.


In [5]:
"""
Cell 3 — DEV threshold selection for the PRIMARY (no-PCA) model.
Sweep 0.10-0.90, pick by DEV F1 only. TEST untouched here.
"""

import numpy as np
from sklearn.metrics import f1_score

dev_proba = model_no_pca.predict_proba(X_dev_scaled)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.01)
best_threshold, best_dev_f1 = 0.5, -1
for t in thresholds:
    preds = (dev_proba >= t).astype(int)
    f1 = f1_score(y_dev, preds, zero_division=0)
    if f1 > best_dev_f1:
        best_dev_f1, best_threshold = f1, t

print(f"Best DEV threshold: {best_threshold:.2f}  (DEV F1 at this threshold: {best_dev_f1:.4f})")

Best DEV threshold: 0.34  (DEV F1 at this threshold: 0.3871)


In [6]:
"""
Cell 4 — Final TEST evaluation, primary (no-PCA) model. TEST touched
exactly once, at the frozen DEV threshold.
"""

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix)

def compute_metrics(y_true, proba, threshold):
    preds = (proba >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, proba),
        "PR_AUC": average_precision_score(y_true, proba),
        "ConfusionMatrix": confusion_matrix(y_true, preds).tolist(),
    }

test_proba = model_no_pca.predict_proba(X_test_scaled)[:, 1]  # TEST used for the first time, right here
dev_metrics = compute_metrics(y_dev, dev_proba, best_threshold)
test_metrics = compute_metrics(y_test, test_proba, best_threshold)

print("DEV  metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in dev_metrics.items()})
print("TEST metrics:", {k: round(v, 4) if isinstance(v, float) else v for k, v in test_metrics.items()})

DEV  metrics: {'Accuracy': 0.3214, 'Precision': 0.24, 'Recall': 1.0, 'F1': 0.3871, 'ROC_AUC': np.float64(0.5038), 'PR_AUC': np.float64(0.3075), 'ConfusionMatrix': [[6, 38], [0, 12]]}
TEST metrics: {'Accuracy': 0.4464, 'Precision': 0.3542, 'Recall': 1.0, 'F1': 0.5231, 'ROC_AUC': np.float64(0.6908), 'PR_AUC': np.float64(0.4744), 'ConfusionMatrix': [[8, 31], [0, 17]]}


In [7]:
"""
Cell 5 — Coefficients / most influential features (no-PCA model, since
PCA components aren't individually interpretable in original feature terms).
"""

import pandas as pd

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model_no_pca.coef_[0],
})
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("Abs_Coefficient", ascending=False).reset_index(drop=True)

print("Top 20 most influential features (by |coefficient|, standardized scale):")
print(coef_df.head(20)[["Feature", "Coefficient"]].to_string(index=False))
print("\n(Positive coefficient -> pushes toward PHQ_Binary=1/depressed; negative -> toward 0)")

coef_df.to_csv("/kaggle/working/logreg_coefficients.csv", index=False)

Top 20 most influential features (by |coefficient|, standardized scale):
                   Feature  Coefficient
    slope500-1500_sma3_max    -0.128926
logRelF0-H1-H2_sma3nz_mean     0.091422
         mfcc3_sma3_median    -0.089468
    F1frequency_sma3nz_p25     0.085061
 logRelF0-H1-H2_sma3nz_std     0.080274
            mfcc4_sma3_max    -0.069586
    F1bandwidth_sma3nz_max     0.067633
            mfcc2_sma3_max    -0.064963
            mfcc2_sma3_p75    -0.064917
  hammarbergIndex_sma3_min    -0.060534
            mfcc3_sma3_min     0.057902
       slope0-500_sma3_min    -0.054861
         mfcc2_sma3_median    -0.053919
 logRelF0-H1-H2_sma3nz_p75     0.049243
            mfcc2_sma3_std    -0.048828
 slope500-1500_sma3_median     0.047303
 logRelF0-H1-H2_sma3nz_min    -0.044663
    F3frequency_sma3nz_min    -0.044161
       alphaRatio_sma3_min    -0.044006
 logRelF0-H1-H2_sma3nz_max     0.043976

(Positive coefficient -> pushes toward PHQ_Binary=1/depressed; negative -> toward 0)


In [8]:
"""
Cell 6 — Compact experiment summary for the BTP results table, including
the no-PCA vs PCA comparison for reference.
"""

import pandas as pd

print("=" * 70)
print("EXPERIMENT SUMMARY — Logistic Regression on 7-stat aggregated eGeMAPS")
print("=" * 70)
print(f"Feature vector length     : {n_features}  (23 eGeMAPS features x 7 stats)")
print(f"Train / Dev / Test counts : {len(train_data)} / {len(dev_data)} / {len(test_data)}")
print(f"Train class distribution  : {train_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Dev class distribution    : {dev_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Test class distribution   : {test_data.PHQ_Binary.value_counts().to_dict()}")
print(f"Best C (no-PCA)           : {params_no_pca['C']}  (train-CV F1: {cvf1_no_pca:.4f})")
print(f"Best C (with-PCA, {pca.n_components_} comps) : {params_pca['C']}  (train-CV F1: {cvf1_pca:.4f})")
print(f"Best DEV threshold        : {best_threshold:.2f}")
print("-" * 70)

summary_df = pd.DataFrame([
    {"Split": "DEV", **{k: v for k, v in dev_metrics.items() if k != "ConfusionMatrix"}},
    {"Split": "TEST", **{k: v for k, v in test_metrics.items() if k != "ConfusionMatrix"}},
])
print(summary_df.round(4).to_string(index=False))
print("-" * 70)
print("DEV  confusion matrix [[TN FP] [FN TP]]:", dev_metrics["ConfusionMatrix"])
print("TEST confusion matrix [[TN FP] [FN TP]]:", test_metrics["ConfusionMatrix"])
print("=" * 70)

summary_df.to_csv("/kaggle/working/logreg_7stat_experiment_summary.csv", index=False)

EXPERIMENT SUMMARY — Logistic Regression on 7-stat aggregated eGeMAPS
Feature vector length     : 161  (23 eGeMAPS features x 7 stats)
Train / Dev / Test counts : 163 / 56 / 56
Train class distribution  : {0: 126, 1: 37}
Dev class distribution    : {0: 44, 1: 12}
Test class distribution   : {0: 39, 1: 17}
Best C (no-PCA)           : 0.01  (train-CV F1: 0.3065)
Best C (with-PCA, 35 comps) : 100  (train-CV F1: 0.3527)
Best DEV threshold        : 0.34
----------------------------------------------------------------------
Split  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
  DEV    0.3214     0.2400     1.0 0.3871   0.5038  0.3075
 TEST    0.4464     0.3542     1.0 0.5231   0.6908  0.4744
----------------------------------------------------------------------
DEV  confusion matrix [[TN FP] [FN TP]]: [[6, 38], [0, 12]]
TEST confusion matrix [[TN FP] [FN TP]]: [[8, 31], [0, 17]]
